# Player words

Associated chart: the player-words scrolly (what she does vs. what is done to her)

Output files:
`player_words.json` in `src/data/`.

Input files: `data/fox_wnba.csv`, `data/espn.csv`, `data/cbs_sports.csv`, `data/usa_today.csv`, `data/just_womens_sports.csv`, `data/the_athletic.csv`.

Requirements: spaCy + scikit-learn (first run ~20 minutes; parses cached in `notebooks/.cache/`)

`pip install spacy scikit-learn`
`python -m spacy download en_core_web_sm`


## Setup

In [ ]:
import csv, html, json, re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import fighting_words as fw

DATA = Path("../data")
OUT = Path("../src/data/player_words.json")
CACHE = Path(".cache"); CACHE.mkdir(exist_ok=True)

MIN_MENTIONS = 40   # attachments a player needs to get her own column
TOP_N = 30
MIN_COUNT = 2   # a word used once is not a comparison; twice, it is a dot
MAX_EXAMPLES = 3

# surname -> canonical; "thomas" (Lia) and "white" (the adjective) stay unmatchable
PLAYERS = {}
for canonical, aliases in {
    "Caitlin Clark": ["caitlin clark", "clark"],
    "Angel Reese": ["angel reese", "reese"],
    "Brittney Griner": ["brittney griner", "griner"],
    "Sophie Cunningham": ["sophie cunningham", "cunningham"],
    "DiJonai Carrington": ["dijonai carrington", "carrington", "dijonai"],
    "Paige Bueckers": ["paige bueckers", "bueckers"],
    "A'ja Wilson": ["a'ja wilson", "aja wilson", "a’ja wilson"],
    "Alyssa Thomas": ["alyssa thomas"],
    "Sue Bird": ["sue bird"],
    "Breanna Stewart": ["breanna stewart", "stewie"],
    "Diana Taurasi": ["diana taurasi", "taurasi"],
    "Chennedy Carter": ["chennedy carter", "chennedy"],
    "Aliyah Boston": ["aliyah boston"],
    "Sabrina Ionescu": ["sabrina ionescu", "ionescu"],
    "Napheesa Collier": ["napheesa collier", "collier", "napheesa"],
    "Natasha Cloud": ["natasha cloud"],
    "Kelsey Plum": ["kelsey plum", "plum"],
    "Candace Parker": ["candace parker"],
    "Kelsey Mitchell": ["kelsey mitchell"],
    "NaLyssa Smith": ["nalyssa smith", "nalyssa"],
    "Jonquel Jones": ["jonquel jones", "jonquel"],
    "Cameron Brink": ["cameron brink"],
    "Sheryl Swoopes": ["sheryl swoopes", "swoopes"],
    "DeWanna Bonner": ["dewanna bonner", "dewanna", "bonner"],
    "JuJu Watkins": ["juju watkins", "juju"],
}.items():
    for alias in aliases:
        PLAYERS[alias] = canonical
NAME_RE = re.compile(r"\b(" + "|".join(sorted(PLAYERS, key=len, reverse=True)) + r")\b", re.I)
# bare surnames that another WNBA player shares; in an article naming her, the
# bare form is left alone (the full name still matches)
LOOKALIKES = {
    "clark": re.compile(r"\balysha clark\b", re.I),
    "collier": re.compile(r"\bcharli collier\b", re.I),
}
PRONOUN = {"she", "her", "hers"}

SKIP_VERBS = {"be", "have", "do", "say", "get", "go", "make", "take", "come", "see",
              "know", "think", "want", "look", "give", "use", "find", "tell", "become"}
SKIP_ADJ = {"other", "more", "most", "such", "same", "own", "many", "much", "few",
            "last", "next", "first", "second", "new", "old", "good", "big", "sure",
            "able", "likely", "several", "former", "recent", "about", "no", "only",
            "right", "free", "late", "key", "current", "third", "fourth", "fifth",
            "sixth", "seventh", "eighth", "ninth", "tenth", "due", "least", "front",
            "whole", "full", "high", "low", "long", "short", "early", "late"}

# nouns that say what someone is ("a villain", "the enforcer", "fan favorite")
# count as descriptions; these ones say nothing
SKIP_NOUN = {"player", "one", "part", "member", "person", "people", "thing",
             "name", "case", "example", "time", "year", "game", "season", "way",
             "pick", "point", "points", "team", "guard", "forward", "center",
             "rookie", "veteran", "teammate", "woman", "women", "girl", "man",
             "reason", "specie", "species", "rebound", "stat", "stats", "knee",
             "ankle", "foot", "hamstring", "quad", "groin", "back", "injury",
             "status", "question", "no", "yes", "day", "night", "week", "month"}
# the pieces of the names themselves must not count as descriptions
NAME_TOKENS = {tok for alias in PLAYERS for tok in alias.replace("'", "").split()} | \
              {tok for alias in PLAYERS for tok in alias.split()}

# Fox staples promo lines mid-paragraph; the all-caps run defeats spaCy's splitter
BOILERPLATE = re.compile(r"CLICK HERE[^a-z]*(?=[A-Z][a-z]|$)")

## Corpora

Headline plus full body for every article with one: Fox as-is; the comparison outlets are the piece's usual five: ESPN, CBS Sports, USA Today, Just Women's Sports, The Athletic.


In [ ]:
csv.field_size_limit(10 ** 9)

def article_text(row):
    # the headline is a sentence of its own, ahead of the body
    head = html.unescape((row.get("headline") or "").strip())
    body = BOILERPLATE.sub(" ", html.unescape(row.get("body") or "")).strip()
    if head and not head.endswith((".", "!", "?")):
        head += "."
    return f"{head} {body}".strip()

def fox_articles():
    for row in csv.DictReader((DATA / "fox_wnba.csv").open()):
        text = article_text(row)
        if len(text.split()) > 80:
            yield {"url": row.get("url", ""), "body": text}

def other_articles():
    for name in ("espn", "cbs_sports", "usa_today", "just_womens_sports", "the_athletic"):
        for row in csv.DictReader((DATA / f"{name}.csv").open()):
            outlet = row.get("outlet") or ""
            text = article_text(row)
            if outlet and not outlet.startswith("Fox") and len(text.split()) > 80:
                yield {"url": row.get("url", ""), "body": text}

fox_rows = list(fox_articles())
other_rows = list(other_articles())
len(fox_rows), len(other_rows)

## Extraction

Dependency parse extraction

In [ ]:
def adjective_form(token):
    # rejoin hyphenated compounds: "all-American" not "american"
    parts, cursor = [token.lemma_.lower()], token
    while (cursor.i >= 2 and cursor.nbor(-1).text == "-"
           and not cursor.nbor(-1).whitespace_ and not cursor.nbor(-2).whitespace_):
        cursor = cursor.nbor(-2)
        parts.insert(0, cursor.text.lower())
    return "-".join(parts)


def window_example(sent, token, budget=460):
    # centre the stored example on the word it is an example OF
    raw = sent.text
    pos = token.idx - sent.start_char
    if len(raw) <= budget:
        piece, pre, post = raw, False, False
    else:
        start = max(0, pos - budget // 2)
        end = min(len(raw), start + budget)
        start = max(0, min(start, end - budget))
        piece, pre, post = raw[start:end], start > 0, end < len(raw)
        if pre:
            piece = piece.split(" ", 1)[-1]
        if post:
            piece = piece.rsplit(" ", 1)[0]
    text = " ".join(html.unescape(piece).split())
    return ("…" if pre else "") + text + ("…" if post else "")


def extract(nlp, rows):
    found = defaultdict(lambda: defaultdict(Counter))
    examples = defaultdict(list)
    seen, texts = set(), []
    for row in rows:
        if row["url"] in seen:
            continue
        seen.add(row["url"])
        texts.append(row["body"])
    print(f"{len(texts)} articles", flush=True)

    for n, doc in enumerate(nlp.pipe(texts, batch_size=20), 1):
        if n % 2000 == 0:
            print(f"  parsed {n}/{len(texts)}", flush=True)
        # an article that names a lookalike ("Alysha Clark") makes the bare
        # surname ambiguous for the whole piece: only the full name counts there
        ambiguous = {alias for alias, rx in LOOKALIKES.items() if rx.search(doc.text)}
        recent = None   # last player named, for pronoun resolution
        for sent in doc.sents:
            # a name can be several tokens ("Alyssa Thomas", "A'ja Wilson"): the
            # match is made on the sentence text and pinned to the span's last
            # token, which is the one the parse attaches to the verb
            named_at = {}
            names = []
            for m in NAME_RE.finditer(sent.text):
                if m.group(1).lower() in ambiguous:
                    continue
                who = PLAYERS[m.group(1).lower()]
                names.append(who)
                span = sent.char_span(m.start(), m.end(), alignment_mode="expand")
                if span is not None:
                    named_at[span[-1].i] = who
            if len(set(names)) > 1 or any(rx.search(sent.text) for rx in LOOKALIKES.values()):
                # two players in one sentence (or a lookalike): attribution is
                # guesswork, so only tokens directly attached to a name count
                recent = None
            elif names:
                recent = names[0]

            for token in sent:
                who = None
                low = token.text.lower()
                if low in PRONOUN:
                    who = recent if len(set(names)) <= 1 else None
                elif token.i in named_at:
                    who = named_at[token.i]
                if not who:
                    continue

                def note(relation, lemma, word_token):
                    found[who][relation][lemma] += 1
                    key = f"{who}|{relation}|{lemma}"
                    if len(examples[key]) < MAX_EXAMPLES:
                        line = window_example(sent, word_token)
                        if all(line != e["t"] for e in examples[key]):
                            examples[key].append({"t": line, "w": word_token.text.lower()})

                # a coordinated name ("Clark and Cunningham were fouled") takes the
                # first conjunct's place in the parse
                anchor = token
                while anchor.dep_ == "conj" and anchor.head is not anchor:
                    anchor = anchor.head
                dep, head = anchor.dep_, anchor.head
                usable = lambda verb: verb.pos_ == "VERB" and verb.lemma_.lower() not in SKIP_VERBS

                # a relative clause on the name: "Cunningham, who punched Clark"
                for clause in token.children:
                    if clause.dep_ == "relcl" and usable(clause):
                        for rel in clause.children:
                            if rel.tag_ in {"WP", "WDT"} or rel.text.lower() in {"who", "whom", "that", "which"}:
                                if rel.dep_ in {"nsubj", "csubj"}:
                                    note("subject", clause.lemma_.lower(), clause)
                                elif rel.dep_ in {"nsubjpass", "dobj", "obj"}:
                                    note("object", clause.lemma_.lower(), clause)

                if dep == "pobj" and head.pos_ == "ADP":
                    verb = head.head
                    if head.dep_ == "agent" and usable(verb):
                        # a passive's doer: "punched in the throat by Alyssa Thomas"
                        note("subject", verb.lemma_.lower(), verb)
                    elif head.dep_ == "prep" and usable(verb):
                        # the verb's target through a preposition: "took a swipe at
                        # Cunningham", "went after Cunningham"
                        note("object", verb.lemma_.lower(), verb)
                    continue
                if head.pos_ in {"VERB", "AUX"} and head.lemma_.lower() not in SKIP_VERBS:
                    if dep in {"nsubj", "csubj", "agent"}:
                        note("subject", head.lemma_.lower(), head)
                    elif dep in {"dobj", "obj", "nsubjpass", "pobj"}:
                        note("object", head.lemma_.lower(), head)

                # JJ/JJR/JJS are real adjectives; VBN is a verb wearing an adjective's hat
                def adjectives_on(noun):
                    for child in noun.children:
                        if child.tag_.startswith("JJ") and child.dep_ == "amod" \
                                and child.lemma_.lower() not in SKIP_ADJ:
                            note("describe", adjective_form(child), child)

                def noun_describes(noun):
                    # "a villain", "the enforcer", "fan favorite": the noun itself
                    lemma = noun.lemma_.lower()
                    if noun.pos_ == "NOUN" and lemma not in SKIP_NOUN and lemma.isalpha() \
                            and noun.text.lower() not in NAME_TOKENS:
                        note("describe", lemma, noun)
                    adjectives_on(noun)

                adjectives_on(token)
                for child in token.children:
                    # "outspoken Fever guard Sophie Cunningham", "Cunningham, the
                    # Fever's fiery guard": the noun beside the name, and its adjectives
                    if child.dep_ in {"compound", "appos", "nmod"} and child.pos_ == "NOUN":
                        noun_describes(child)
                if dep == "appos" and head.pos_ == "NOUN":
                    # "the fan favorite, Sophie Cunningham"
                    noun_describes(head)
                if dep in {"nsubj", "nsubjpass"} and head.pos_ == "AUX":
                    for child in head.children:
                        if child.dep_ in {"acomp", "attr"}:
                            if child.tag_.startswith("JJ"):
                                if child.lemma_.lower() not in SKIP_ADJ:
                                    note("describe", adjective_form(child), child)
                            elif child.pos_ == "NOUN":
                                # "Cunningham is a villain"
                                noun_describes(child)
    return found, dict(examples)

In [ ]:
def cached_parse(name, rows):
    cache = CACHE / f"{name}.json"
    if cache.exists():
        raw = json.loads(cache.read_text())
        print(f"{name}: reusing {cache} (delete it to re-parse)")
        return ({p: {r: Counter(w) for r, w in b.items()} for p, b in raw["counts"].items()},
                raw["examples"])
    import spacy
    nlp = spacy.load("en_core_web_sm", disable=["ner"])
    found, examples = extract(nlp, rows)
    cache.write_text(json.dumps({
        "counts": {p: {r: dict(w) for r, w in b.items()} for p, b in found.items()},
        "examples": examples}))
    return found, examples

found, examples = cached_parse("fox_parsed", fox_rows)
other_found, other_examples = cached_parse("others_parsed", other_rows)

## Participles

spaCy's small model tags passive participles as adjectives, so "she was violently fouled." Trying to correct. 

In [ ]:
TRUE_PARTICIPIAL_ADJ = {
    "excited", "worried", "concerned", "disappointed", "frustrated", "injured",
    "retired", "undefeated", "unfazed", "beloved", "talented", "gifted",
    "experienced", "respected", "celebrated", "grateful", "tired", "surprised",
    "determined", "committed", "qualified", "unranked", "ranked", "seeded",
}

def stems_of(word):
    # fouled -> foul, shoved -> shove, slammed -> slam
    base = word.rsplit("-", 1)[-1]
    if not base.endswith("ed"):
        return []
    out = {base[:-2], base[:-1]}
    if len(base) > 4 and base[-3] == base[-4]:
        out.add(base[:-3])
    return [w for w in out if len(w) > 2]


def reclassify_participles(found, examples):
    verbs = set()
    for buckets in found.values():
        for relation in ("subject", "object"):
            verbs.update(buckets.get(relation, Counter()))

    moved = Counter()
    for player, buckets in found.items():
        describe = buckets.get("describe")
        if not describe:
            continue
        for word in list(describe):
            if word in TRUE_PARTICIPIAL_ADJ or not word.endswith("ed"):
                continue
            if re.search(r"-(hand|foot|side|sight|mind|will|heart)ed$", word):
                continue   # "right-handed" describes a person
            if not any(stem in verbs for stem in stems_of(word)):
                continue
            count = describe.pop(word)
            buckets.setdefault("object", Counter())[word] += count
            moved[word] += count
            old, new = f"{player}|describe|{word}", f"{player}|object|{word}"
            if old in examples:
                examples.setdefault(new, [])
                for line in examples.pop(old):
                    if line not in examples[new]:
                        examples[new].append(line)
    if moved:
        print("moved to 'done to her': "
              + ", ".join(f"{w} ({n})" for w, n in moved.most_common(8)))
    return found, examples

found, examples = reclassify_participles(found, examples)
other_found, other_examples = reclassify_participles(other_found, other_examples)

## Weighted log-odds



In [ ]:
def contrast(subject, others, min_count=None):
    from sklearn.feature_extraction.text import CountVectorizer
    min_count = MIN_COUNT if min_count is None else min_count

    left = [" ".join([word] * n) for word, n in subject.items()]
    right = [" ".join([word] * n) for word, n in others.items()]
    if not left or not right:
        return {"clark": [], "others": []}
    vectorizer = CountVectorizer(decode_error="ignore", min_df=1, max_df=1.0)
    scores = fw.bayes_compare_language(left, right, ngram=1, prior=0.05, cv=vectorizer)
    total = subject + others
    kept = sorted([(w, z) for w, z in scores if total[w] >= min_count], key=lambda p: p[1])
    shape = lambda p: {"word": p[0], "z": round(p[1], 2),
                       "clark": subject.get(p[0], 0), "others": others.get(p[0], 0)}
    # every word above the floor goes in; the TOP_N strongest per side are
    # flagged for labels (and carry examples)
    mine = [shape(p) for p in kept[::-1] if p[1] > 0 and subject.get(p[0], 0) > 0]
    theirs = [shape(p) for p in kept if p[1] < 0 and others.get(p[0], 0) > 0]
    for side in (mine, theirs):
        for i, word in enumerate(side):
            word["top"] = i < TOP_N
    return {"clark": mine, "others": theirs}


def find_examples(store, who, relation, word):
    parts = ("subject", "object", "describe") if relation == "all" else (relation,)
    out = []
    for part in parts:
        out.extend(store.get(f"{who}|{part}|{word}", []))
    return out[:MAX_EXAMPLES]

## Build the payload


In [ ]:
RELATIONS = ("subject", "object", "describe", "all")

def bucket(store, name, relation):
    if relation != "all":
        return store.get(name, {}).get(relation, Counter())
    merged = Counter()
    for part in ("subject", "object", "describe"):
        merged.update(store.get(name, {}).get(part, Counter()))
    return merged

mentions = {name: sum(sum(c.values()) for c in b.values()) for name, b in found.items()}
cast = [n for n, total in sorted(mentions.items(), key=lambda kv: -kv[1])
        if total >= MIN_MENTIONS]
print(f"{len(cast)} players clear {MIN_MENTIONS} attachments ({len(found) - len(cast)} below)")

players = {}
for name in cast:
    entry = {"relations": {}, "counts": {}}
    for relation in RELATIONS:
        mine = bucket(found, name, relation)
        others = Counter()
        for peer in found:
            if peer != name:
                others.update(bucket(found, peer, relation))
        scored = contrast(mine, others, None)
        for side, owner in (("clark", name), ("others", None)):
            for word in scored[side]:
                if not word.get("top"):
                    continue
                if owner:
                    word["examples"] = find_examples(examples, owner, relation, word["word"])
                else:
                    # pooled column: examples from whichever peer uses the word most
                    best, count = None, 0
                    for peer, buckets in found.items():
                        if peer == name:
                            continue
                        got = buckets.get(relation, Counter()).get(word["word"], 0)
                        if got > count:
                            best, count = peer, got
                    word["examples"] = examples.get(
                        f"{best}|{relation}|{word['word']}", []) if best else []
                    word["who"] = best
        entry["relations"][relation] = scored
        entry["counts"][relation] = {"clark": sum(mine.values()),
                                     "others": sum(others.values())}

        theirs = bucket(other_found, name, relation)
        if sum(mine.values()) >= 25 and sum(theirs.values()) >= 25:
            versus = contrast(mine, theirs, None)
            for word in versus["clark"]:
                if word.get("top"):
                    word["examples"] = find_examples(examples, name, relation, word["word"])
            for word in versus["others"]:
                if word.get("top"):
                    word["examples"] = find_examples(other_examples, name, relation, word["word"])
            entry.setdefault("vsOutlets", {})[relation] = versus
            entry.setdefault("vsCounts", {})[relation] = {
                "clark": sum(mine.values()), "others": sum(theirs.values())}
    players[name] = entry

In [ ]:
PAIRS = [("Caitlin Clark", "Angel Reese")]
pairs = {}
for left, right in PAIRS:
    if left not in found or right not in found:
        continue
    block = {"relations": {}, "counts": {}}
    for relation in RELATIONS:
        a = bucket(found, left, relation)
        b = bucket(found, right, relation)
        if sum(a.values()) < 25 or sum(b.values()) < 25:
            continue
        scored = contrast(a, b)
        for word in scored["clark"]:
            if word.get("top"):
                word["examples"] = find_examples(examples, left, relation, word["word"])
        for word in scored["others"]:
            if word.get("top"):
                word["examples"] = find_examples(examples, right, relation, word["word"])
        block["relations"][relation] = scored
        block["counts"][relation] = {"clark": sum(a.values()), "others": sum(b.values())}
    pairs[f"{left} vs {right}"] = {**block, "left": left, "right": right}

payload = {"order": cast, "mentions": {n: mentions[n] for n in cast},
           "pairs": pairs,
           "players": players,
           "hasOutletComparison": bool(other_found),
           "source": "article bodies",
           "nDocs": len(fox_rows)}

#### Overwrites `src/data/player_words.json`

In [ ]:
# slim copy for the site: one example survives per word (hover), the rest dropped
slim = json.loads(json.dumps(payload))
# the player-versus-other-players block is not on the site; keep only its labelled words
for block in slim["players"].values():
    for scored in block["relations"].values():
        for side in ("clark", "others"):
            scored[side] = [w for w in scored[side] if w.get("top")]
for block in list(slim["players"].values()) + list(slim["pairs"].values()):
    for group in ("relations", "vsOutlets"):
        for scored in (block.get(group) or {}).values():
            for side in ("clark", "others"):
                for word in scored[side]:
                    examples_here = word.pop("examples", None) or []
                    if examples_here:
                        first = examples_here[0]
                        if isinstance(first, dict):
                            word["eg"] = first["t"]
                            word["egw"] = first["w"]
                        else:
                            word["eg"] = first

OUT.write_text(json.dumps(slim))
print(f"wrote {OUT} ({OUT.stat().st_size / 1e6:.1f} MB)")

## Preview

In [ ]:
new = json.loads(OUT.read_text())
preview = []
for name in new["order"]:
    rels = new["players"][name]["relations"]
    preview.append({
        "player": name,
        "mentions": new["mentions"][name],
        "she does": ", ".join(w["word"] for w in rels["subject"]["clark"][:5]),
        "done to her": ", ".join(w["word"] for w in rels["object"]["clark"][:5]),
        "described as": ", ".join(w["word"] for w in rels["describe"]["clark"][:5]),
    })
pd.DataFrame(preview)

## Spot check

In [ ]:
import random

player = random.choice(list(payload["players"]))
rel = random.choice(["subject", "object", "describe"])
side = payload["players"][player]["relations"].get(rel, {}).get("clark", [])
print(f"— {player} · {rel} ('she does' / 'done to her' / 'described as')")
for w in random.sample(side, min(3, len(side))):
    print(f"   {w['word']} ({w.get('count', '?')})")
    for ex in (w.get("examples") or [])[:2]:
        text = ex["t"] if isinstance(ex, dict) else ex
        print(f"      “{str(text)[:100]}”")